This notebook takes the base Palaeohispanic dataset and processes it to acquire a usable dataset for the classification of the dual system.

In [1]:
import pandas as pd
import numpy as np
import re

1. Remove FALSA and SUSPECTA entries
2. Filter entries by Iberian (LEVANTINO) and Celtiberian signaries, because those are the ones where the dual system exists
3. Remove LATINA and GRAECA entries, because they do not present the dual system
4. Detect listing items and split the entries to have manageable blocks of text.
    - This may introduce a problem. After splitting, one part of the text might contain the dual system explicit evidence while the other part might not, but the classifier still needs to output the dual class.

In [2]:
data = pd.read_csv('./out/palaeohispanic_dataset.csv')

In [3]:
data = data[~data['refHesperia'].str.contains('FALSA|SUSPECTA')]

In [4]:
data = data[data['signary_cat'].isin(['LEVANTINO','CELTIBERICO E.','CELTIBERICO','CELTIBERICO W.'])]

In [5]:
data = data[~data['refHesperia'].str.contains('LATINA|GRAECA')]

In [6]:
data['clean_text'] = data['clean_text'].str.split('\(?[a-zA-Z]\)',regex=True)
data = data.explode('clean_text')
data['clean_text'] = data['clean_text'].str.split('(?:I+V?-)?\d+?\.',regex=True)
data = data.explode('clean_text')
data['clean_text'] = data['clean_text'].str.split('[A-D][0-9I]+[a-z]?[0-9]?:|[A-D]:',regex=True)
data = data.explode('clean_text')
data['clean_text'] = data['clean_text'].str.split('I+-.+?\.\s',regex=True)
data = data.explode('clean_text')
data['clean_text'] = data['clean_text'].str.split('B-I+:\s',regex=True)
data = data.explode('clean_text')
data['clean_text'] = data['clean_text'].str.split('^[AB][0-9]?\)?\s|\s[AB][0-9]?\)?\s',regex=True)
data = data.explode('clean_text')
data['clean_text'] = data['clean_text'].str.split('[a-zA-Z][0-9]:?',regex=True)
data = data.explode('clean_text')
data['clean_text'] = data['clean_text'].str.split('[\(A-Za-z]*[0-9]\)',regex=True)
data = data.explode('clean_text')
data['clean_text'] = data['clean_text'].str.split('\sI{2,3}:',regex=True)
data = data.explode('clean_text')
data['clean_text'] = data['clean_text'].str.split('\d+:',regex=True)
data = data.explode('clean_text')

In [7]:
data = data[data['clean_text'].str.strip().str.len() > 2]

In [8]:
data['clean_text'] = data['clean_text'].str.replace('\s+',' ',regex=True)
data['clean_text'] = data['clean_text'].str.strip()

In [9]:
# set([v for row in data['clean_text'].str.extractall('(\d+:)').values for v in row])

Remove text that should not be here:

- A mixed inscription that contains "ME" in Latin or only Greek characters (: ΔΔΔ Π)
- Latin text still in some inscriptions

In [10]:
data['clean_text'] = data['clean_text'].str.replace('ME','',regex=True)
data['clean_text'] = data['clean_text'].str.replace(': ΔΔΔ Π','',regex=True)
data = data[data['clean_text']!='']

In [11]:
print(data['clean_text'].str.split('').explode().value_counts().to_dict())

{'': 3870, 'a': 3784, 'i': 3758, 'e': 2786, '-': 2333, 'r': 2248, 'k': 2223, 't': 2215, '́': 2125, 'u': 1971, 's': 1924, 'b': 1701, 'n': 1652, ':': 1631, 'o': 1445, ' ': 1406, ']': 1355, '[': 1277, 'l': 1167, '+': 1048, '̲': 785, 'm': 739, '̱': 507, 'I': 472, 'd': 378, '̣': 331, 'g': 330, 'z': 153, '·': 48, 'Π': 28, 'L': 27, 'V': 22, '.': 19, '̌': 12, '̂': 10, '*': 10, 'S': 7, '̠': 6, 'E': 5, 'X': 4, 'A': 4, '̇': 3, 'T': 3, 'O': 3, '>': 2, 'F': 2, 'N': 2, 'R': 1, 'C': 1, 'H': 1, 'Y': 1, '(': 1, ')': 1, 'Q': 1, '̄': 1, '∙': 1, '<': 1}


In [12]:
# for v in data[data['clean_text'].str.contains('\(')][['refHesperia','clean_text']].values:
#     print('# ' + v[0] + ' ### ' + v[1])

In [13]:
LATIN_FRAGMENTS = [
    'HEIC · EST · SITVS +A',
    'FVLVIA · LINTEARIA',
    'NEI',
    'Q:OFELI:',
    'XII',
    'O',
    'Y',
    '(',
    ')'
]

for lf in LATIN_FRAGMENTS:
    data['clean_text'] = data['clean_text'].str.replace(re.escape(lf),'',regex=True)

The non-dual inscriptions' transcriptions have a problem. They use 't' and 'k' as default since the pronunciation is usually unknown. These characters are always the simple variants. However, in the dual transcriptions, the simple graphemes are always transcribed as the voiced consonants 'd' and 'g'. Therefore, there is a contradiction. We change t→d and k→g in all the non-dual inscriptions.

In [14]:
data.loc[data['dual_system_cat']=='NO DUAL','clean_text'] = data.loc[data['dual_system_cat']=='NO DUAL','clean_text'].str.replace('t','d',regex=True)
data.loc[data['dual_system_cat']=='NO DUAL','clean_text'] = data.loc[data['dual_system_cat']=='NO DUAL','clean_text'].str.replace('k','g',regex=True)

In [15]:
data = data[data['clean_text']!='']
data = data[~data['clean_text'].str.match('^\W$')]
data.shape

(1934, 36)

We need to fill NULL values:

- `municipality_latitude` & `longitude`: mean values
- `dating_min` & `max` to min and max values respectively
  - Recalculate dating_mean and width

In [16]:
data['municipality_latitude'] = data['municipality_latitude'].fillna(data['municipality_latitude'].mean())
data['municipality_longitude'] = data['municipality_longitude'].fillna(data['municipality_longitude'].mean())
data['dating_min'] = data['dating_min'].fillna(data['dating_min'].min())
data['dating_max'] = data['dating_max'].fillna(data['dating_max'].max())
data['dating_mean'] = (data['dating_max'] + data['dating_min']) / 2
data['dating_width'] = data['dating_max'] - data['dating_min']


And filter the dual system entries that are null

In [17]:
print(data.shape)
data = data[~data['clean_text'].str.match('^\W+$')]
print(data.shape)

(1934, 36)
(1927, 36)


In [18]:
print(data.shape)
data = data[~data['dual_system_cat'].isna()]
print(data.shape)

(1927, 36)
(1926, 36)


In [19]:
data['dual_system_cat'].value_counts(normalize=True)

dual_system_cat
NO DUAL    0.727414
DUAL       0.272586
Name: proportion, dtype: float64

In [ ]:
data.to_csv('./out/dual_system_dataset.csv',index=False)